# 03 -- Data Cleaning
**Purpose:** Fix all data quality issues identified in Notebook 01.
Produce a clean, analysis-ready dataset as the input to feature engineering.

**Input:** `data/raw/transactions.csv`
**Output:** `data/processed/transactions_clean.parquet`

## Learning Objectives
- Apply the two-type NaN framework (Not Applicable vs Truly Missing)
- Write imputation logic with strategy documentation
- Handle dtype conversions programmatically
- Understand why parquet is preferred over CSV for ML pipelines

## Business Context
Every missing value is a decision. A missing `fema_purpose_code` on an export transaction
is a compliance risk -- it may indicate a deliberate omission to avoid FEMA scrutiny.
A missing `failure_reason` on a completed transaction is expected -- the field does not apply.
Treating both identically is wrong and will introduce noise into the model.

## Cleaning Checklist (from Notebook 01 findings)

| Column | Issue | Treatment |
|---|---|---|
| sanction_list_source | 100% missing | Drop -- encoded in counterparty_sanction_flag |
| fraud_type | 99.5% missing | Leave as NaN -- not available pre-detection |
| fema_purpose_code | 4.03% missing | Impute by business_type modal code + create fema_missing_flag |
| beneficiary_bank_country | 3.09% missing | Impute from beneficiary_country |
| invoice_match_flag | 9.35% missing | Create has_invoice flag, impute 0 |
| timestamp | object dtype | Convert to datetime64 |
| gst_registered | zero variance | Drop |

## Step 1 -- Before You Clean
For each column in the checklist above:
1. What would happen to the ML model if we imputed it incorrectly?
2. Which column carries the highest compliance risk if cleaned wrong?
3. Why do we create a flag BEFORE imputing, not after?

Answer in a markdown cell before writing code.

## Step 2 -- Drop Columns
Drop `sanction_list_source` and `gst_registered`.
Print shape before and after. Verify `counterparty_sanction_flag` still exists.

## Step 3 -- Fix dtypes
Convert `timestamp` to datetime64. Print dtype before and after.
Extract: `txn_year`, `txn_month`, `txn_day` as integer columns.

## Step 4 -- Impute Missing Values
For each truly-missing column: create the binary flag first, then impute.
Document your imputation strategy as a comment above each block.

## Step 5 -- Validate
After cleaning: print df.isnull().sum() -- only Not-Applicable columns should have NaN.
Assert fraud rate is unchanged. Print shape.

## Step 6 -- Save
Save to `data/processed/transactions_clean.parquet`.
In a markdown cell, explain: why parquet instead of CSV for ML pipelines?

## Definition of Done
- [ ] All Not-Applicable NaN columns: left as NaN with explanation
- [ ] All Truly-Missing columns: imputed with strategy documented
- [ ] Zero-variance and leakage columns dropped
- [ ] timestamp converted to datetime64
- [ ] Fraud rate unchanged after cleaning
- [ ] Output saved to parquet
- [ ] You can explain every decision without notes

In [89]:
import pandas as pd
import numpy as np  
transactions = pd.read_csv('../data/raw/transactions.csv')

C:\Users\Admin\AppData\Local\Temp\ipykernel_12440\2726862700.py:3: DtypeWarning: Columns (46) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv('../data/raw/transactions.csv')


In [90]:
print(transactions.columns[46])


sanction_list_source


In [91]:
transactions.describe()

,hour_of_day,day_of_week,month,amount_usd,amount_inr,business_age_years,declared_annual_turnover_inr,kyc_verified,gst_registered,has_iec,...,has_supporting_doc,is_round_number,firc_generated,time_since_last_login_hrs,is_new_device,is_foreign_ip,is_failed,is_reversed,retry_count,is_fraud
count,50800.000000,50800.000000,50800.000000,50800.000000,5.080000e+04,50800.000000,5.080000e+04,50800.000000,50800.0,50800.000000,...,50100.000000,50800.000000,49697.000000,44617.000000,50800.000000,50800.000000,50800.000000,50800.000000,50800.000000,50800.000000
mean,12.987933,2.991752,3.680177,25436.459883,2.108868e+06,10.614606,1.279966e+08,0.906319,1.0,0.472638,...,0.896607,0.001043,0.430026,24.077283,0.081890,0.030177,0.029114,0.006398,0.028012,0.004921
std,4.235618,2.005335,1.524661,19783.526772,1.849604e+06,5.714478,1.191890e+08,0.291387,0.0,0.499256,...,0.304475,0.032284,0.495084,13.766653,0.274199,0.171076,0.168128,0.079730,0.226545,0.069980
min,0.000000,0.000000,1.000000,1002.210000,2.277021e+04,1.000000,1.793225e+06,0.000000,1.0,0.000000,...,0.000000,0.000000,0.000000,0.080000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,10.000000,1.000000,3.000000,10902.072500,7.903072e+05,6.000000,3.439522e+07,1.000000,1.0,0.000000,...,1.000000,0.000000,0.000000,12.300000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,13.000000,3.000000,4.000000,19490.795000,1.544151e+06,11.000000,8.835551e+07,1.000000,1.0,0.000000,...,1.000000,0.000000,0.000000,24.100000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,16.000000,5.000000,5.000000,35397.020000,2.800745e+06,16.000000,1.921687e+08,1.000000,1.0,1.000000,...,1.000000,0.000000,1.000000,36.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,23.000000,6.000000,6.000000,200000.000000,2.102000e+07,20.000000,4.996054e+08,1.000000,1.0,1.000000,...,1.000000,1.000000,1.000000,48.000000,1.000000,1.000000,1.000000,1.000000,3.000000,1.000000


In [92]:
transactions.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50800 entries, 0 to 50799
Data columns (total 61 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   transaction_id                50800 non-null  object 
 1   timestamp                     50800 non-null  object 
 2   hour_of_day                   50800 non-null  int64  
 3   day_of_week                   50800 non-null  int64  
 4   month                         50800 non-null  int64  
 5   transaction_type              50800 non-null  object 
 6   transaction_status            50800 non-null  object 
 7   amount_usd                    50800 non-null  float64
 8   amount_inr                    50800 non-null  float64
 9   currency                      50800 non-null  object 
 10  payment_rail                  50800 non-null  object 
 11  fema_purpose_code             48753 non-null  object 
 12  customer_id                   50800 non-null  object 
 13  b

In [93]:
transactions.shape

(50800, 61)

In [94]:
transactions = transactions.drop(columns=['gst_registered'])


In [95]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50800 entries, 0 to 50799
Data columns (total 60 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   transaction_id                50800 non-null  object 
 1   timestamp                     50800 non-null  object 
 2   hour_of_day                   50800 non-null  int64  
 3   day_of_week                   50800 non-null  int64  
 4   month                         50800 non-null  int64  
 5   transaction_type              50800 non-null  object 
 6   transaction_status            50800 non-null  object 
 7   amount_usd                    50800 non-null  float64
 8   amount_inr                    50800 non-null  float64
 9   currency                      50800 non-null  object 
 10  payment_rail                  50800 non-null  object 
 11  fema_purpose_code             48753 non-null  object 
 12  customer_id                   50800 non-null  object 
 13  b

In [96]:
transactions.timestamp = pd.to_datetime(transactions['timestamp'], format='%Y-%m-%d %H:%M:%S')

In [97]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50800 entries, 0 to 50799
Data columns (total 60 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   transaction_id                50800 non-null  object        
 1   timestamp                     50800 non-null  datetime64[ns]
 2   hour_of_day                   50800 non-null  int64         
 3   day_of_week                   50800 non-null  int64         
 4   month                         50800 non-null  int64         
 5   transaction_type              50800 non-null  object        
 6   transaction_status            50800 non-null  object        
 7   amount_usd                    50800 non-null  float64       
 8   amount_inr                    50800 non-null  float64       
 9   currency                      50800 non-null  object        
 10  payment_rail                  50800 non-null  object        
 11  fema_purpose_code           

In [98]:
df = transactions.copy()

In [99]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50800 entries, 0 to 50799
Data columns (total 60 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   transaction_id                50800 non-null  object        
 1   timestamp                     50800 non-null  datetime64[ns]
 2   hour_of_day                   50800 non-null  int64         
 3   day_of_week                   50800 non-null  int64         
 4   month                         50800 non-null  int64         
 5   transaction_type              50800 non-null  object        
 6   transaction_status            50800 non-null  object        
 7   amount_usd                    50800 non-null  float64       
 8   amount_inr                    50800 non-null  float64       
 9   currency                      50800 non-null  object        
 10  payment_rail                  50800 non-null  object        
 11  fema_purpose_code           

In [100]:
df.isnull().sum()


transaction_id                      0
timestamp                           0
hour_of_day                         0
day_of_week                         0
month                               0
transaction_type                    0
transaction_status                  0
amount_usd                          0
amount_inr                          0
currency                            0
payment_rail                        0
fema_purpose_code                2047
customer_id                         0
business_type                       0
industry_sector                     0
business_city                       0
business_state                      0
business_region                     0
business_age_years                  0
incorporation_type                  0
employee_count_range                0
annual_revenue_tier                 0
declared_annual_turnover_inr        0
rbi_risk_category                   0
kyc_verified                        0
has_iec                             0
is_opc      

In [101]:
transactions['fema_purpose_code'].isnull().sum()

2047

In [102]:

transactions['fema_missing_flag'] = transactions['fema_purpose_code'].isnull().astype(int)


In [103]:
transactions['fema_missing_flag'].sum()


2047

In [104]:
transactions['fema_purpose_code'].isnull().sum()


2047

In [105]:
# fema_purpose_code has 4.03% missing values

# Step 1: Create the flag BEFORE imputing
# This records WHICH rows had missing data — that information disappears after imputation
# A missing FEMA code on an export transaction is itself a fraud signal
df['fema_missing_flag'] = df['fema_purpose_code'].isna().astype(int)

# Step 2: Impute using the most common code for that business_type
# Logic: a business with type "EXPORTER" always uses the same FEMA code
# So we fill the gap with whatever code that business_type uses most often
modal_fema = df.groupby('business_type')['fema_purpose_code'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'UNKNOWN')
)
df['fema_purpose_code'] = df['fema_purpose_code'].fillna(modal_fema)

print("fema_purpose_code NaN remaining:", df['fema_purpose_code'].isna().sum())
print("fema_missing_flag — rows flagged:", df['fema_missing_flag'].sum())


fema_purpose_code NaN remaining: 0
fema_missing_flag — rows flagged: 2047


In [106]:
# beneficiary_bank_country has 3.09% missing values

# The logic here is simple: if we know the beneficiary's country,
# the bank is almost certainly in the same country.
# So we fill the missing bank country directly from beneficiary_country.
# No flag needed — this is a data entry gap, not a fraud signal.

df['beneficiary_bank_country'] = df['beneficiary_bank_country'].fillna(df['beneficiary_country'])

print("beneficiary_bank_country NaN remaining:", df['beneficiary_bank_country'].isna().sum())


beneficiary_bank_country NaN remaining: 0


In [107]:
# invoice_match_flag has 9.35% missing values

# Step 1: Create has_invoice flag BEFORE imputing
# Missing invoice_match_flag means no invoice was submitted at all
# has_invoice = 1 means an invoice was submitted (match or no match)
# has_invoice = 0 means nothing was submitted — this IS a fraud signal
# (High-value cross-border payments with no invoice are a TBML red flag)
df['has_invoice'] = df['invoice_match_flag'].notna().astype(int)

# Step 2: Impute with 0
# If no invoice was submitted, there is nothing to match — so match flag = 0
df['invoice_match_flag'] = df['invoice_match_flag'].fillna(0)

print("invoice_match_flag NaN remaining:", df['invoice_match_flag'].isna().sum())
print("has_invoice — transactions WITH invoice:", df['has_invoice'].sum())
print("has_invoice — transactions WITHOUT invoice:", (df['has_invoice'] == 0).sum())


invoice_match_flag NaN remaining: 0
has_invoice — transactions WITH invoice: 46048
has_invoice — transactions WITHOUT invoice: 4752


In [108]:
# Check 1: Null count across all columns
# After cleaning, only fraud_type should have NaN (it is a Not-Applicable column — left intentionally)
print("=== COLUMNS WITH NaN ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0])

# Check 2: Fraud rate must be unchanged
# Cleaning must never alter the target column (is_fraud)
# If the fraud rate changed, something went wrong
print("\n=== FRAUD RATE CHECK ===")
print(f"Fraud rate: {df['is_fraud'].mean():.5f}  (expected: 0.00492)")

# Check 3: Shape — confirm row count is still 50,800
# We should have lost 2 columns (sanction_list_source, gst_registered)
# and gained 3 new columns (fema_missing_flag, has_invoice + timestamp parts if added)
print("\n=== SHAPE ===")
print(df.shape)


=== COLUMNS WITH NaN ===
sanction_list_source         50798
has_supporting_doc             700
firc_generated                1103
time_since_last_login_hrs     6183
failure_reason               49321
reversal_reason              50475
fraud_type                   50550
dtype: int64

=== FRAUD RATE CHECK ===
Fraud rate: 0.00492  (expected: 0.00492)

=== SHAPE ===
(50800, 62)


In [109]:
# Drop sanction_list_source — it was missed in Section 2
df = df.drop(columns=['sanction_list_source'])

print("sanction_list_source still present:", 'sanction_list_source' in df.columns)
print("New shape:", df.shape)


sanction_list_source still present: False
New shape: (50800, 61)


In [110]:
# --- time_since_last_login_hrs ---
# Flag FIRST: missing login time = new customer or no login history
# A new account initiating a large cross-border transfer is a fraud signal
df['login_missing_flag'] = df['time_since_last_login_hrs'].isna().astype(int)

# Then impute with median
median_login = df['time_since_last_login_hrs'].median()
df['time_since_last_login_hrs'] = df['time_since_last_login_hrs'].fillna(median_login)

print(f"time_since_last_login_hrs NaN remaining: {df['time_since_last_login_hrs'].isna().sum()}")
print(f"login_missing_flag — rows flagged: {df['login_missing_flag'].sum()}")
print(f"Median used: {median_login:.2f} hrs")


time_since_last_login_hrs NaN remaining: 0
login_missing_flag — rows flagged: 6183
Median used: 24.10 hrs


In [111]:
# --- has_supporting_doc ---
# Flag first: missing supporting document on a cross-border payment is a fraud signal
df['supporting_doc_missing_flag'] = df['has_supporting_doc'].isna().astype(int)
df['has_supporting_doc'] = df['has_supporting_doc'].fillna(0)

print(f"has_supporting_doc NaN remaining: {df['has_supporting_doc'].isna().sum()}")
print(f"supporting_doc_missing_flag — rows flagged: {df['supporting_doc_missing_flag'].sum()}")


has_supporting_doc NaN remaining: 0
supporting_doc_missing_flag — rows flagged: 700


In [112]:
# --- firc_generated ---
# FIRC is only issued after a successful inward remittance completes
# Missing = not yet generated or not applicable for this transaction type
# No fraud signal in the missingness — impute 0 directly
df['firc_generated'] = df['firc_generated'].fillna(0)

print(f"firc_generated NaN remaining: {df['firc_generated'].isna().sum()}")


firc_generated NaN remaining: 0


In [113]:
print("=== COLUMNS WITH NaN ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0])

print("\n=== FRAUD RATE CHECK ===")
print(f"Fraud rate: {df['is_fraud'].mean():.5f}  (expected: 0.00492)")

print("\n=== SHAPE ===")
print(df.shape)


=== COLUMNS WITH NaN ===
failure_reason     49321
reversal_reason    50475
fraud_type         50550
dtype: int64

=== FRAUD RATE CHECK ===
Fraud rate: 0.00492  (expected: 0.00492)

=== SHAPE ===
(50800, 63)


In [114]:
import os

# Create the processed folder if it does not exist
os.makedirs('../data/processed', exist_ok=True)

# Save as parquet — preserves dtypes, compressed, fast to reload
df.to_parquet('../data/processed/transactions_clean.parquet', index=False)

# Verify the saved file loads correctly with dtypes intact
df_check = pd.read_parquet('../data/processed/transactions_clean.parquet')

print("Saved and reloaded successfully.")
print("Shape:", df_check.shape)
print("timestamp dtype:", df_check['timestamp'].dtype)
print("NaN columns after reload:", df_check.isnull().sum()[df_check.isnull().sum() > 0].index.tolist())


Saved and reloaded successfully.
Shape: (50800, 63)
timestamp dtype: datetime64[ns]
NaN columns after reload: ['failure_reason', 'reversal_reason', 'fraud_type']


In [115]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/transactions_clean.parquet')
print("Shape:", df.shape)
print("Columns:", df.shape[1])


Shape: (50800, 63)
Columns: 63


In [117]:
# Log transformation of transaction amount
# +1 avoids log(0) error if any amount is zero
df['log_amount_usd'] = np.log1p(df['amount_usd'])

# Show original vs transformed side by side
print(df[['transaction_id', 'amount_usd', 'log_amount_usd']].head(10).to_string())


  transaction_id  amount_usd  log_amount_usd
0   4a20bcb3-c08    27854.24       10.234776
1   7c22a3f6-e30    10270.49        9.237127
2   4009e080-70a    47718.10       10.773087
3   2ea35f08-4c7    27086.55       10.206829
4   18e8b44c-5d7     9693.66        9.179330
5   55f9616d-b5f    10585.20        9.267307
6   da02d14e-818    24275.75       10.097274
7   1f91e3c4-7d3    58079.22       10.969580
8   47d2d8b9-9b0     7588.47        8.934517
9   b75da643-30a    26723.82       10.193348


In [120]:
# Extract hour of day (0–23) and day of week (0=Monday, 6=Sunday) from timestamp
df['submission_hour'] = df['timestamp'].dt.hour
df['submission_day']  = df['timestamp'].dt.dayofweek

# Show a sample
print(df[['transaction_id', 'timestamp', 'submission_hour', 'submission_day']].head(10).to_string())

# Quick distribution check — what hours are most common?
print("\nTop 5 submission hours:")
print(df['submission_hour'].value_counts().head())


  transaction_id           timestamp  submission_hour  submission_day
0   4a20bcb3-c08 2026-02-03 07:27:42                7               1
1   7c22a3f6-e30 2026-03-15 10:17:22               10               6
2   4009e080-70a 2026-05-28 17:14:30               17               3
3   2ea35f08-4c7 2026-03-25 06:37:01                6               2
4   18e8b44c-5d7 2026-05-19 03:58:00                3               1
5   55f9616d-b5f 2026-03-28 15:12:04               15               5
6   da02d14e-818 2026-06-06 18:22:23               18               5
7   1f91e3c4-7d3 2026-04-07 11:23:43               11               1
8   47d2d8b9-9b0 2026-04-21 13:36:45               13               1
9   b75da643-30a 2026-04-12 13:03:21               13               6

Top 5 submission hours:
submission_hour
11    4661
12    4543
13    4511
14    4455
10    4387
Name: count, dtype: int64


In [121]:
# Flag transactions within 10% below the USD 10,000 threshold
# i.e., amount between 9,000 and 10,000
# These are potential structuring attempts

THRESHOLD = 10000
BAND = 0.10  # 10% below threshold

df['is_near_threshold'] = (
    (df['amount_usd'] >= THRESHOLD * (1 - BAND)) &
    (df['amount_usd'] <  THRESHOLD)
).astype(int)

print("Transactions near threshold:", df['is_near_threshold'].sum())
print("As % of total:", f"{df['is_near_threshold'].mean()*100:.2f}%")

# Do near-threshold transactions have a higher fraud rate?
print("\nFraud rate — near threshold:", df[df['is_near_threshold']==1]['is_fraud'].mean().round(4))
print("Fraud rate — all others:    ", df[df['is_near_threshold']==0]['is_fraud'].mean().round(4))


Transactions near threshold: 1516
As % of total: 2.98%

Fraud rate — near threshold: 0.0132
Fraud rate — all others:     0.0047


In [122]:
print(df.columns.tolist())


['transaction_id', 'timestamp', 'hour_of_day', 'day_of_week', 'month', 'transaction_type', 'transaction_status', 'amount_usd', 'amount_inr', 'currency', 'payment_rail', 'fema_purpose_code', 'customer_id', 'business_type', 'industry_sector', 'business_city', 'business_state', 'business_region', 'business_age_years', 'incorporation_type', 'employee_count_range', 'annual_revenue_tier', 'declared_annual_turnover_inr', 'rbi_risk_category', 'kyc_verified', 'has_iec', 'is_opc', 'director_count', 'is_msme_registered', 'dispute_count_90d', 'failed_txn_count_30d', 'has_prior_fraud_flag', 'account_id', 'account_type', 'account_age_days', 'is_dormant_account', 'days_since_account_last_used', 'is_primary_account', 'beneficiary_id', 'beneficiary_country', 'beneficiary_type', 'beneficiary_bank_country', 'is_new_beneficiary', 'is_new_country', 'counterparty_sanction_flag', 'invoice_match_flag', 'has_supporting_doc', 'is_round_number', 'firc_generated', 'time_since_last_login_hrs', 'is_new_device', 'is

In [123]:
# is_off_hours: transaction submitted before 9am or after 6pm
df['is_off_hours'] = (
    (df['submission_hour'] < 9) | (df['submission_hour'] > 18)
).astype(int)

# is_weekend: day_of_week 5=Saturday, 6=Sunday
df['is_weekend'] = (df['submission_day'] >= 5).astype(int)

print("Off-hours transactions:", df['is_off_hours'].sum(), f"({df['is_off_hours'].mean()*100:.1f}%)")
print("Weekend transactions:  ", df['is_weekend'].sum(),   f"({df['is_weekend'].mean()*100:.1f}%)")

print("\nFraud rate — off hours:", df[df['is_off_hours']==1]['is_fraud'].mean().round(4))
print("Fraud rate — in hours: ", df[df['is_off_hours']==0]['is_fraud'].mean().round(4))

print("\nFraud rate — weekend:  ", df[df['is_weekend']==1]['is_fraud'].mean().round(4))
print("Fraud rate — weekday:  ", df[df['is_weekend']==0]['is_fraud'].mean().round(4))


Off-hours transactions: 11674 (23.0%)
Weekend transactions:   14526 (28.6%)

Fraud rate — off hours: 0.0062
Fraud rate — in hours:  0.0045

Fraud rate — weekend:   0.005
Fraud rate — weekday:   0.0049


In [124]:
print("Payment rail values:")
print(df['payment_rail'].value_counts())

print("\nTop 10 beneficiary countries:")
print(df['beneficiary_country'].value_counts().head(10))


Payment rail values:
payment_rail
SWIFT      46052
Fedwire     1709
CHIPS       1437
CHAPS       1006
SEPA         596
Name: count, dtype: int64

Top 10 beneficiary countries:
beneficiary_country
US           10112
UK            8006
Germany       7736
UAE           7193
Australia     6379
Singapore     5308
Canada        2952
China         2173
Malaysia       896
Myanmar         10
Name: count, dtype: int64


In [125]:
# Define which countries are valid for each restricted rail
FEDWIRE_CHIPS_COUNTRIES = {'US'}
CHAPS_COUNTRIES         = {'UK'}
SEPA_COUNTRIES          = {'Germany', 'France', 'Netherlands', 'Italy', 'Spain',
                            'Belgium', 'Austria', 'Sweden', 'Denmark', 'Finland'}

def is_mismatch(row):
    rail    = row['payment_rail']
    country = row['beneficiary_country']

    if rail in ('Fedwire', 'CHIPS') and country not in FEDWIRE_CHIPS_COUNTRIES:
        return 1
    if rail == 'CHAPS' and country not in CHAPS_COUNTRIES:
        return 1
    if rail == 'SEPA' and country not in SEPA_COUNTRIES:
        return 1
    return 0

df['rail_corridor_mismatch'] = df.apply(is_mismatch, axis=1)

print("Mismatch transactions:", df['rail_corridor_mismatch'].sum())
print("As % of total:", f"{df['rail_corridor_mismatch'].mean()*100:.2f}%")

print("\nFraud rate — mismatch:    ", df[df['rail_corridor_mismatch']==1]['is_fraud'].mean().round(4))
print("Fraud rate — no mismatch: ", df[df['rail_corridor_mismatch']==0]['is_fraud'].mean().round(4))


Mismatch transactions: 941
As % of total: 1.85%

Fraud rate — mismatch:     0.0032
Fraud rate — no mismatch:  0.005


In [126]:
# Load customers and accounts
customers = pd.read_csv('../data/raw/customers.csv')
accounts  = pd.read_csv('../data/raw/accounts.csv')

print("=== CUSTOMERS ===")
print("Shape:", customers.shape)
print(customers.columns.tolist())

print("\n=== ACCOUNTS ===")
print("Shape:", accounts.shape)
print(accounts.columns.tolist())


=== CUSTOMERS ===
Shape: (500, 31)
['customer_id', 'business_name', 'cin_number', 'business_type', 'industry_sector', 'incorporation_type', 'business_city', 'business_state', 'business_region', 'business_age_years', 'employee_count_range', 'founder_profile', 'iec_number', 'has_iec', 'kyc_verified', 'kyc_date', 'gst_registered', 'rbi_risk_category', 'declared_annual_turnover_inr', 'annual_revenue_tier', 'primary_currency', 'declared_countries', 'primary_bank', 'bank_relationship_age_years', 'director_count', 'is_opc', 'is_msme_registered', 'msme_category', 'dispute_count_90d', 'failed_txn_count_30d', 'has_prior_fraud_flag']

=== ACCOUNTS ===
Shape: (780, 13)
['account_id', 'customer_id', 'account_type', 'account_currency', 'account_bank', 'account_age_days', 'account_opened_date', 'last_used_date', 'is_primary', 'account_status', 'is_dormant', 'days_since_last_used', 'balance_tier']


In [127]:
# Join accounts to get account_currency — only column we need that isn't already in df
df = df.merge(
    accounts[['account_id', 'account_currency']],
    on='account_id',
    how='left'
)

print("Shape after join:", df.shape)
print("account_currency sample:")
print(df[['transaction_id', 'currency', 'account_currency']].head(10).to_string())


Shape after join: (50800, 72)
account_currency sample:
  transaction_id currency account_currency
0   4a20bcb3-c08      GBP              GBP
1   7c22a3f6-e30      GBP              GBP
2   4009e080-70a      GBP              GBP
3   2ea35f08-4c7      USD              USD
4   18e8b44c-5d7      EUR              EUR
5   55f9616d-b5f      AED              AED
6   da02d14e-818      USD              USD
7   1f91e3c4-7d3      AED              AED
8   47d2d8b9-9b0      AED              AED
9   b75da643-30a      USD              USD


In [128]:
# 1 = currencies match (normal), 0 = mismatch (unusual)
df['currency_match'] = (df['currency'] == df['account_currency']).astype(int)

print("Currency matches:   ", df['currency_match'].sum())
print("Currency mismatches:", (df['currency_match'] == 0).sum())

print("\nFraud rate — match:    ", df[df['currency_match']==1]['is_fraud'].mean().round(4))
print("Fraud rate — mismatch: ", df[df['currency_match']==0]['is_fraud'].mean().round(4))


Currency matches:    50800
Currency mismatches: 0

Fraud rate — match:     0.0049


AttributeError: 'float' object has no attribute 'round'

In [ ]:
# as there is zero mismatches exist
df = df.drop(columns=['currency_match'])
print("currency_match dropped. Shape:", df.shape)


currency_match dropped. Shape: (50800, 72)


In [130]:
# Check what values exist first
print("RBI risk categories:")
print(df['rbi_risk_category'].value_counts())


RBI risk categories:
rbi_risk_category
Low       37313
Medium    12157
High       1330
Name: count, dtype: int64


In [131]:
# Map text categories to ordered numbers: Low=0, Medium=1, High=2
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['rbi_risk_score'] = df['rbi_risk_category'].map(risk_map)

print("rbi_risk_score distribution:")
print(df['rbi_risk_score'].value_counts().sort_index())

print("\nFraud rate by risk score:")
print(df.groupby('rbi_risk_score')['is_fraud'].mean().round(4))


rbi_risk_score distribution:
rbi_risk_score
0    37313
1    12157
2     1330
Name: count, dtype: int64

Fraud rate by risk score:
rbi_risk_score
0    0.0049
1    0.0049
2    0.0060
Name: is_fraud, dtype: float64


In [132]:
# Map text categories to ordered numbers: Low=0, Medium=1, High=2
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['rbi_risk_score'] = df['rbi_risk_category'].map(risk_map)

print("rbi_risk_score distribution:")
print(df['rbi_risk_score'].value_counts().sort_index())

print("\nFraud rate by risk score:")
print(df.groupby('rbi_risk_score')['is_fraud'].mean().round(4))


rbi_risk_score distribution:
rbi_risk_score
0    37313
1    12157
2     1330
Name: count, dtype: int64

Fraud rate by risk score:
rbi_risk_score
0    0.0049
1    0.0049
2    0.0060
Name: is_fraud, dtype: float64


In [133]:
# days_since_incorporation: newer companies taking large cross-border positions = higher risk
df['days_since_incorporation'] = (df['business_age_years'] * 365).astype(int)

# turnover_to_txn_ratio: what fraction of annual revenue is this single transaction?
# Using INR/INR — no FX conversion, no training-serving skew
df['turnover_to_txn_ratio'] = (df['amount_inr'] / df['declared_annual_turnover_inr']).round(4)

print(df[['amount_inr', 'declared_annual_turnover_inr', 'turnover_to_txn_ratio', 'days_since_incorporation']].head(10).to_string())

print("\nFraud rate by turnover_to_txn_ratio quartile:")
df['_ratio_quartile'] = pd.qcut(df['turnover_to_txn_ratio'], q=4, labels=['Q1','Q2','Q3','Q4'])
print(df.groupby('_ratio_quartile', observed=True)['is_fraud'].mean().round(4))
df = df.drop(columns=['_ratio_quartile'])


   amount_inr  declared_annual_turnover_inr  turnover_to_txn_ratio  days_since_incorporation
0  2927480.62                      37218081                 0.0787                      4015
1  1079428.50                     131683194                 0.0082                      5110
2  5015172.31                     112090177                 0.0447                      4745
3  2261726.92                     178827258                 0.0126                      7300
4   874368.13                     261841778                 0.0033                      1095
5   240495.74                       4856278                 0.0495                      2190
6  2027025.12                      66733961                 0.0304                      3285
7  1319559.88                     258132749                 0.0051                      4380
8   172410.04                      23212339                 0.0074                      6570
9  2231438.97                     247196246                 0.0090    

In [134]:
# Check what business types exist
print("Business types:")
print(df['business_type'].value_counts())


Business types:
business_type
Exporter       13666
IT Services    10426
Importer       10344
EdTech          7538
SME             4514
SaaS            4312
Name: count, dtype: int64


In [135]:
# One-hot encode business_type — creates one 0/1 column per category
# drop_first=True drops one category to avoid redundancy
# (if all other columns are 0, we know it must be the dropped category)
business_dummies = pd.get_dummies(df['business_type'], prefix='biz', drop_first=True)
df = pd.concat([df, business_dummies], axis=1)

print("New columns added:")
print(business_dummies.columns.tolist())
print("\nSample:")
print(df[['business_type'] + business_dummies.columns.tolist()].head(6).to_string())


New columns added:
['biz_Exporter', 'biz_IT Services', 'biz_Importer', 'biz_SME', 'biz_SaaS']

Sample:
  business_type  biz_Exporter  biz_IT Services  biz_Importer  biz_SME  biz_SaaS
0   IT Services         False             True         False    False     False
1      Exporter          True            False         False    False     False
2      Exporter          True            False         False    False     False
3   IT Services         False             True         False    False     False
4      Importer         False            False          True    False     False
5           SME         False            False         False     True     False


In [136]:
biz_cols = ['biz_Exporter', 'biz_IT Services', 'biz_Importer', 'biz_SME', 'biz_SaaS']
df[biz_cols] = df[biz_cols].astype(int)

print(df[['business_type'] + biz_cols].head(6).to_string())


  business_type  biz_Exporter  biz_IT Services  biz_Importer  biz_SME  biz_SaaS
0   IT Services             0                1             0        0         0
1      Exporter             1                0             0        0         0
2      Exporter             1                0             0        0         0
3   IT Services             0                1             0        0         0
4      Importer             0                0             1        0         0
5           SME             0                0             0        1         0


In [137]:
# Sort by customer and timestamp — required for all behavioural calculations
df = df.sort_values(['customer_id', 'timestamp']).reset_index(drop=True)

print("Sorted. Shape:", df.shape)
print(df[['customer_id', 'timestamp', 'amount_usd']].head(10).to_string())


Sorted. Shape: (50800, 80)
  customer_id           timestamp  amount_usd
0       C0000 2026-01-02 18:37:04     3200.93
1       C0000 2026-01-03 19:41:04    12049.47
2       C0000 2026-01-04 11:39:13    18561.90
3       C0000 2026-01-06 21:30:15    15420.34
4       C0000 2026-01-08 10:03:09    16148.07
5       C0000 2026-01-09 10:53:06    26835.03
6       C0000 2026-01-15 12:12:42    26907.08
7       C0000 2026-01-20 07:45:29    19266.56
8       C0000 2026-01-20 16:48:18    18382.76
9       C0000 2026-01-31 20:50:40     2115.28


In [138]:
# Set timestamp as the index — required for pandas time-based rolling windows
df = df.set_index('timestamp')

# For each transaction, count how many transactions this customer
# made in the 24 hours BEFORE this one (closed='left' excludes the current row)
df['txn_count_24h'] = (
    df.groupby('customer_id')['amount_usd']
    .rolling('24h', closed='left')
    .count()
    .reset_index(level=0, drop=True)
    .fillna(0)
    .astype(int)
)

# Restore timestamp as a regular column
df = df.reset_index()

print(df[['customer_id', 'timestamp', 'amount_usd', 'txn_count_24h']].head(12).to_string())
print("\nMax transactions seen in 24h window:", df['txn_count_24h'].max())
print("Fraud rate — 3+ txns in 24h:", df[df['txn_count_24h'] >= 3]['is_fraud'].mean().round(4))
print("Fraud rate — 0 txns in 24h:  ", df[df['txn_count_24h'] == 0]['is_fraud'].mean().round(4))


   customer_id           timestamp  amount_usd  txn_count_24h
0        C0000 2026-01-02 18:37:04     3200.93              0
1        C0000 2026-01-03 19:41:04    12049.47              0
2        C0000 2026-01-04 11:39:13    18561.90              1
3        C0000 2026-01-06 21:30:15    15420.34              0
4        C0000 2026-01-08 10:03:09    16148.07              0
5        C0000 2026-01-09 10:53:06    26835.03              0
6        C0000 2026-01-15 12:12:42    26907.08              0
7        C0000 2026-01-20 07:45:29    19266.56              0
8        C0000 2026-01-20 16:48:18    18382.76              1
9        C0000 2026-01-31 20:50:40     2115.28              0
10       C0000 2026-02-03 15:58:23     7147.38              0
11       C0000 2026-02-07 17:39:58    18882.05              0

Max transactions seen in 24h window: 7
Fraud rate — 3+ txns in 24h: 0.0016
Fraud rate — 0 txns in 24h:   0.0048


In [139]:
df = df.set_index('timestamp')

df['txn_count_7d'] = (
    df.groupby('customer_id')['amount_usd']
    .rolling('7D', closed='left')
    .count()
    .reset_index(level=0, drop=True)
    .fillna(0)
    .astype(int)
)

df = df.reset_index()

print(df[['customer_id', 'timestamp', 'txn_count_24h', 'txn_count_7d']].head(12).to_string())
print("\nFraud rate — 5+ txns in 7d:", df[df['txn_count_7d'] >= 5]['is_fraud'].mean().round(4))
print("Fraud rate — 0 txns in 7d: ", df[df['txn_count_7d'] == 0]['is_fraud'].mean().round(4))


   customer_id           timestamp  txn_count_24h  txn_count_7d
0        C0000 2026-01-02 18:37:04              0             0
1        C0000 2026-01-03 19:41:04              0             1
2        C0000 2026-01-04 11:39:13              1             2
3        C0000 2026-01-06 21:30:15              0             3
4        C0000 2026-01-08 10:03:09              0             4
5        C0000 2026-01-09 10:53:06              0             5
6        C0000 2026-01-15 12:12:42              0             1
7        C0000 2026-01-20 07:45:29              0             1
8        C0000 2026-01-20 16:48:18              1             2
9        C0000 2026-01-31 20:50:40              0             0
10       C0000 2026-02-03 15:58:23              0             1
11       C0000 2026-02-07 17:39:58              0             2

Fraud rate — 5+ txns in 7d: 0.0046
Fraud rate — 0 txns in 7d:  0.0036


In [ ]:
# checks 30days zscore

df = df.set_index('timestamp')

# Rolling 30-day mean per customer (excluding current transaction)
rolling_mean_30d = (
    df.groupby('customer_id')['amount_usd']
    .rolling('30D', closed='left')
    .mean()
    .reset_index(level=0, drop=True)
)

# Rolling 30-day standard deviation per customer
rolling_std_30d = (
    df.groupby('customer_id')['amount_usd']
    .rolling('30D', closed='left')
    .std()
    .reset_index(level=0, drop=True)
)

# Z-score: how many standard deviations away from the customer's 30d average
# Where std is 0 or NaN (first transaction), fill with 0 — no anomaly signal possible
df['amount_zscore_30d'] = (
    (df['amount_usd'] - rolling_mean_30d) / rolling_std_30d
).fillna(0).replace([np.inf, -np.inf], 0).round(4)

df = df.reset_index()

print(df[['customer_id', 'timestamp', 'amount_usd', 'amount_zscore_30d']].head(12).to_string())
print("\nFraud rate — zscore > 2:", df[df['amount_zscore_30d'] > 2]['is_fraud'].mean().round(4))
print("Fraud rate — zscore < 2:", df[df['amount_zscore_30d'] <= 2]['is_fraud'].mean().round(4))


   customer_id           timestamp  amount_usd  amount_zscore_30d
0        C0000 2026-01-02 18:37:04     3200.93             0.0000
1        C0000 2026-01-03 19:41:04    12049.47             0.0000
2        C0000 2026-01-04 11:39:13    18561.90             1.7480
3        C0000 2026-01-06 21:30:15    15420.34             0.5382
4        C0000 2026-01-08 10:03:09    16148.07             0.5793
5        C0000 2026-01-09 10:53:06    26835.03             2.2963
6        C0000 2026-01-15 12:12:42    26907.08             1.4862
7        C0000 2026-01-20 07:45:29    19266.56             0.2703
8        C0000 2026-01-20 16:48:18    18382.76             0.1400
9        C0000 2026-01-31 20:50:40     2115.28            -2.1098
10       C0000 2026-02-03 15:58:23     7147.38            -1.2792
11       C0000 2026-02-07 17:39:58    18882.05             0.2065

Fraud rate — zscore > 2: 0.1371
Fraud rate — zscore < 2: 0.002


In [141]:
#checks 90days zscore


df = df.set_index('timestamp')

rolling_mean_90d = (
    df.groupby('customer_id')['amount_usd']
    .rolling('90D', closed='left')
    .mean()
    .reset_index(level=0, drop=True)
)

rolling_std_90d = (
    df.groupby('customer_id')['amount_usd']
    .rolling('90D', closed='left')
    .std()
    .reset_index(level=0, drop=True)
)

df['amount_zscore_90d'] = (
    (df['amount_usd'] - rolling_mean_90d) / rolling_std_90d
).fillna(0).replace([np.inf, -np.inf], 0).round(4)

df = df.reset_index()

print(df[['customer_id', 'timestamp', 'amount_usd', 'amount_zscore_30d', 'amount_zscore_90d']].head(12).to_string())
print("\nFraud rate — 90d zscore > 2:", df[df['amount_zscore_90d'] > 2]['is_fraud'].mean().round(4))


   customer_id           timestamp  amount_usd  amount_zscore_30d  amount_zscore_90d
0        C0000 2026-01-02 18:37:04     3200.93             0.0000             0.0000
1        C0000 2026-01-03 19:41:04    12049.47             0.0000             0.0000
2        C0000 2026-01-04 11:39:13    18561.90             1.7480             1.7480
3        C0000 2026-01-06 21:30:15    15420.34             0.5382             0.5382
4        C0000 2026-01-08 10:03:09    16148.07             0.5793             0.5793
5        C0000 2026-01-09 10:53:06    26835.03             2.2963             2.2963
6        C0000 2026-01-15 12:12:42    26907.08             1.4862             1.4862
7        C0000 2026-01-20 07:45:29    19266.56             0.2703             0.2703
8        C0000 2026-01-20 16:48:18    18382.76             0.1400             0.1400
9        C0000 2026-01-31 20:50:40     2115.28            -2.1098            -2.1098
10       C0000 2026-02-03 15:58:23     7147.38            -1.2792

In [ ]:
# its not runing becouse the synthetic data max gap is only 24.86 days



# For each transaction, calculate days since the customer's PREVIOUS transaction
# shift(1) gives the previous row's timestamp within each customer group
df['prev_txn_timestamp'] = df.groupby('customer_id')['timestamp'].shift(1)

df['days_since_last_txn'] = (
    (df['timestamp'] - df['prev_txn_timestamp'])
    .dt.total_seconds() / 86400
).round(2)

# First transaction per customer has no previous — fill with -1 as a marker
df['days_since_last_txn'] = df['days_since_last_txn'].fillna(-1)

# Drop the helper column
df = df.drop(columns=['prev_txn_timestamp'])

print(df[['customer_id', 'timestamp', 'days_since_last_txn']].head(12).to_string())
print("\nFraud rate — gap > 30 days:", df[df['days_since_last_txn'] > 30]['is_fraud'].mean().round(4))
print("Fraud rate — gap <= 30 days:", df[df['days_since_last_txn'] <= 30]['is_fraud'].mean().round(4))


   customer_id           timestamp  days_since_last_txn
0        C0000 2026-01-02 18:37:04                -1.00
1        C0000 2026-01-03 19:41:04                 1.04
2        C0000 2026-01-04 11:39:13                 0.67
3        C0000 2026-01-06 21:30:15                 2.41
4        C0000 2026-01-08 10:03:09                 1.52
5        C0000 2026-01-09 10:53:06                 1.03
6        C0000 2026-01-15 12:12:42                 6.06
7        C0000 2026-01-20 07:45:29                 4.81
8        C0000 2026-01-20 16:48:18                 0.38
9        C0000 2026-01-31 20:50:40                11.17
10       C0000 2026-02-03 15:58:23                 2.80
11       C0000 2026-02-07 17:39:58                 4.07


AttributeError: 'float' object has no attribute 'round'

In [143]:
print("Gap distribution (days):")
print(df[df['days_since_last_txn'] > 0]['days_since_last_txn'].describe().round(2))

print("\nFraud rate — gap > 7 days:", df[df['days_since_last_txn'] > 7]['is_fraud'].mean().round(4))
print("Fraud rate — gap <= 7 days:", df[df['days_since_last_txn'] <= 7]['is_fraud'].mean().round(4))
print("Fraud rate — first txn (-1):", df[df['days_since_last_txn'] == -1]['is_fraud'].mean().round(4))


Gap distribution (days):
count    49983.00
mean         1.76
std          1.88
min          0.01
25%          0.50
50%          1.11
75%          2.30
max         24.86
Name: days_since_last_txn, dtype: float64

Fraud rate — gap > 7 days: 0.0043
Fraud rate — gap <= 7 days: 0.0049
Fraud rate — first txn (-1): 0.002


In [144]:
# Flag transactions in the last 2 weeks of a quarter-end month
# Quarter-end months: March (3), June (6), September (9), December (12)

QUARTER_END_MONTHS = {3, 6, 9, 12}

df['quarter_end_flag'] = (
    (df['timestamp'].dt.month.isin(QUARTER_END_MONTHS)) &
    (df['timestamp'].dt.day >= 15)
).astype(int)

print("Quarter-end transactions:", df['quarter_end_flag'].sum(), f"({df['quarter_end_flag'].mean()*100:.1f}%)")
print("\nFraud rate — quarter end:", df[df['quarter_end_flag']==1]['is_fraud'].mean().round(4))
print("Fraud rate — non quarter:", df[df['quarter_end_flag']==0]['is_fraud'].mean().round(4))


Quarter-end transactions: 10908 (21.5%)

Fraud rate — quarter end: 0.0054
Fraud rate — non quarter: 0.0048


In [145]:
# FATF risk tier per beneficiary country
# 0 = Low    — FATF member, strong AML framework
# 1 = Medium — Monitored or recently removed from grey list
# 2 = High   — Elevated AML risk, limited FATF oversight
# 3 = Very High — On FATF blacklist or active grey list

FATF_SCORES = {
    'US'        : 0,  # FATF founding member, strong controls
    'UK'        : 0,  # FATF member, strong controls
    'Germany'   : 0,  # FATF member, strong controls
    'Australia' : 0,  # FATF member, strong controls
    'Canada'    : 0,  # FATF member, strong controls
    'Singapore' : 0,  # FATF member, strong AML regime
    'UAE'       : 1,  # Removed from grey list 2024 — still monitored
    'Malaysia'  : 1,  # Removed from grey list 2023 — still monitored
    'China'     : 2,  # Not blacklisted but elevated risk, opacity concerns
    'Myanmar'   : 3,  # FATF blacklist — highest AML risk
}

df['corridor_fatf_score'] = df['beneficiary_country'].map(FATF_SCORES).fillna(1)

print("FATF score distribution:")
print(df['corridor_fatf_score'].value_counts().sort_index())

print("\nFraud rate by FATF score:")
print(df.groupby('corridor_fatf_score')['is_fraud'].mean().round(4))


FATF score distribution:
corridor_fatf_score
0.0    40493
1.0     8124
2.0     2173
3.0       10
Name: count, dtype: int64

Fraud rate by FATF score:
corridor_fatf_score
0.0    0.0041
1.0    0.0080
2.0    0.0037
3.0    1.0000
Name: is_fraud, dtype: float64


In [146]:
print("Total columns:", len(df.columns))
print("\nAll columns:")
for col in sorted(df.columns):
    print(f"  {col}")


Total columns: 87

All columns:
  account_age_days
  account_currency
  account_id
  account_type
  amount_inr
  amount_usd
  amount_zscore_30d
  amount_zscore_90d
  annual_revenue_tier
  beneficiary_bank_country
  beneficiary_country
  beneficiary_id
  beneficiary_type
  biz_Exporter
  biz_IT Services
  biz_Importer
  biz_SME
  biz_SaaS
  business_age_years
  business_city
  business_region
  business_state
  business_type
  corridor_fatf_score
  counterparty_sanction_flag
  currency
  customer_id
  day_of_week
  days_since_account_last_used
  days_since_incorporation
  days_since_last_txn
  declared_annual_turnover_inr
  director_count
  dispute_count_90d
  employee_count_range
  failed_txn_count_30d
  failure_reason
  fema_missing_flag
  fema_purpose_code
  firc_generated
  fraud_type
  has_iec
  has_invoice
  has_prior_fraud_flag
  has_supporting_doc
  hour_of_day
  incorporation_type
  industry_sector
  invoice_match_flag
  is_dormant_account
  is_failed
  is_foreign_ip
  is_fraud

In [147]:
# Columns to drop — raw sources, identifiers, duplicates, leakage, NaN columns
COLUMNS_TO_DROP = [
    # Identifiers — not features
    'transaction_id', 'customer_id', 'account_id', 'beneficiary_id',

    # Raw datetime — we extracted what we need
    'timestamp',

    # Leakage and Not-Applicable
    'fraud_type', 'failure_reason', 'reversal_reason',

    # Duplicates — engineered versions already exist
    'amount_usd',                    # → log_amount_usd
    'amount_inr',                    # → turnover_to_txn_ratio
    'hour_of_day',                   # → submission_hour
    'day_of_week',                   # → submission_day
    'submission_dow',                # duplicate of submission_day
    'business_age_years',            # → days_since_incorporation
    'declared_annual_turnover_inr',  # → turnover_to_txn_ratio
    'rbi_risk_category',             # → rbi_risk_score
    'business_type',                 # → biz_* dummies

    # Geographic raw — engineered versions exist
    'beneficiary_country',           # → corridor_fatf_score
    'beneficiary_bank_country',
    'business_city', 'business_state', 'business_region',

    # Payment metadata raw — engineered versions exist
    'payment_rail',                  # → rail_corridor_mismatch
    'currency', 'account_currency',
    'fema_purpose_code',             # → fema_missing_flag

    # Categorical columns not encoded — too many categories for this PoC
    'beneficiary_type', 'incorporation_type',
    'employee_count_range', 'annual_revenue_tier',
    'industry_sector', 'account_type',
    'transaction_status', 'transaction_type',
]

# Build feature matrix X and target y
X = df.drop(columns=COLUMNS_TO_DROP + ['is_fraud'])
y = df['is_fraud']

print("=== FEATURE MATRIX ===")
print("Shape:", X.shape)
print("\nFeatures:", X.columns.tolist())
print("\nNaN count:", X.isnull().sum().sum())
print("\nFraud cases in y:", y.sum(), f"({y.mean()*100:.3f}%)")


=== FEATURE MATRIX ===
Shape: (50800, 52)

Features: ['month', 'kyc_verified', 'has_iec', 'is_opc', 'director_count', 'is_msme_registered', 'dispute_count_90d', 'failed_txn_count_30d', 'has_prior_fraud_flag', 'account_age_days', 'is_dormant_account', 'days_since_account_last_used', 'is_primary_account', 'is_new_beneficiary', 'is_new_country', 'counterparty_sanction_flag', 'invoice_match_flag', 'has_supporting_doc', 'is_round_number', 'firc_generated', 'time_since_last_login_hrs', 'is_new_device', 'is_foreign_ip', 'is_failed', 'is_reversed', 'retry_count', 'fema_missing_flag', 'has_invoice', 'login_missing_flag', 'supporting_doc_missing_flag', 'log_amount_usd', 'submission_hour', 'submission_day', 'is_near_threshold', 'is_off_hours', 'is_weekend', 'rail_corridor_mismatch', 'rbi_risk_score', 'days_since_incorporation', 'turnover_to_txn_ratio', 'biz_Exporter', 'biz_IT Services', 'biz_Importer', 'biz_SME', 'biz_SaaS', 'txn_count_24h', 'txn_count_7d', 'amount_zscore_30d', 'amount_zscore_90d

In [148]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Save feature matrix and target separately
X.to_parquet('../data/processed/feature_matrix.parquet', index=False)
y.to_frame().to_parquet('../data/processed/target.parquet', index=False)

# Verify
X_check = pd.read_parquet('../data/processed/feature_matrix.parquet')
y_check = pd.read_parquet('../data/processed/target.parquet')

print("feature_matrix.parquet — shape:", X_check.shape)
print("target.parquet         — shape:", y_check.shape)
print("Fraud cases in target: ", y_check['is_fraud'].sum())
print("\nNotebook 04 complete.")


feature_matrix.parquet — shape: (50800, 52)
target.parquet         — shape: (50800, 1)
Fraud cases in target:  250

Notebook 04 complete.
